In [ ]:
!pip install openmeteo-requests pandas requests-cache retry-requests

In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
from pathlib import Path
import time

## Crawl OpenMeteo

In [ ]:
# Cấu hình session
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Danh sách trạm
stations = [
    {"station_id": "48820099999", "name": "Hà Nội (Nội Bài)", "lat": 21.221192, "lon": 105.807178},
    {"station_id": "48823099999", "name": "Hà Nội (Láng)", "lat": 20.433333, "lon": 106.150000},
    {"station_id": "48826099999", "name": "Hải Phòng (Phù Liễn)", "lat": 20.800000, "lon": 106.633333},
    {"station_id": "48831099999", "name": "Nam Định", "lat": 21.600000, "lon": 105.833333},
    {"station_id": "48840099999", "name": "Thanh Hóa", "lat": 19.750000, "lon": 105.783333},
    {"station_id": "48845099999", "name": "Nghệ An (Vinh)", "lat": 18.736725, "lon": 105.670881},
    {"station_id": "48848099999", "name": "Thừa Thiên Huế (Phú Bài)", "lat": 17.483333, "lon": 106.600000},
    {"station_id": "48852099999", "name": "Đà Nẵng", "lat": 16.401500, "lon": 107.702614},
    {"station_id": "48863099999", "name": "Bình Định (Quy Nhơn)", "lat": 15.133333, "lon": 108.783333},
    {"station_id": "48866099999", "name": "Lâm Đồng (Đà Lạt)", "lat": 14.004522, "lon": 108.017158},
    {"station_id": "48870099999", "name": "Khánh Hòa (Nha Trang)", "lat": 13.766667, "lon": 109.216667},
    {"station_id": "48894099999", "name": "Đắk Lắk (Buôn Ma Thuột)", "lat": 10.650000, "lon": 106.716667},
    {"station_id": "48900099999", "name": "TP. Hồ Chí Minh (Tân Sơn Nhất)", "lat": 10.818797, "lon": 106.651856},
    {"station_id": "48907099999", "name": "Cần Thơ", "lat": 10.000000, "lon": 105.083333},
    {"station_id": "48914099999", "name": "Cà Mau", "lat": 9.183333, "lon": 105.150000},
]

# Danh sách biến DAILY (Cập nhật từ yêu cầu của bạn)
daily_vars = [
    "precipitation_sum", "rain_sum", "showers_sum", "snowfall_sum", 
    "snowfall_water_equivalent_sum", "precipitation_hours", "precipitation_probability_max", 
    "precipitation_probability_mean", "precipitation_probability_min",
    "cape_mean", "cape_max", "cape_min", "updraft_max",
    "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean", 
    "apparent_temperature_max", "apparent_temperature_min", "apparent_temperature_mean", 
    "dew_point_2m_max", "dew_point_2m_min", "dew_point_2m_mean", 
    "wet_bulb_temperature_2m_max", "wet_bulb_temperature_2m_min", "wet_bulb_temperature_2m_mean",
    "relative_humidity_2m_max", "relative_humidity_2m_min", "relative_humidity_2m_mean", 
    "vapour_pressure_deficit_max", "leaf_wetness_probability_mean",
    "pressure_msl_mean", "pressure_msl_max", "pressure_msl_min", "surface_pressure_mean",
    "wind_speed_10m_max", "wind_gusts_10m_max", "wind_direction_10m_dominant", 
    "wind_speed_10m_mean", "wind_speed_10m_min", "wind_gusts_10m_mean", "wind_gusts_10m_min",
    "shortwave_radiation_sum", "cloud_cover_mean", "cloud_cover_max", "cloud_cover_min", 
    "sunshine_duration", "daylight_duration",
    "et0_fao_evapotranspiration", "soil_moisture_0_to_7cm_mean", "soil_moisture_0_to_10cm_mean", 
    "soil_moisture_7_to_28cm_mean", "soil_moisture_28_to_100cm_mean", "soil_moisture_0_to_100cm_mean", 
    "soil_temperature_0_to_7cm_mean", "soil_temperature_0_to_100cm_mean",
    "weather_code", "visibility_mean", "visibility_min"
]

url = "https://archive-api.open-meteo.com/v1/archive"
all_df = []
start_time = time.time()

print("--- BẮT ĐẦU CRAWL OPEN-METEO (DAILY) ---\n")

for st in stations:
    print(f"Đang tải: {st['name']} ...")
    params = {
        "latitude": st["lat"],
        "longitude": st["lon"],
        "start_date": "2015-01-01",
        "end_date": "2024-12-31",
        "daily": daily_vars,
        "timezone": "Asia/Bangkok"
    }

    try:
        responses = openmeteo.weather_api(url, params=params)
        response = responses[0]
        daily = response.Daily()

        # Tạo DataFrame từ dữ liệu hàng ngày
        daily_data = {"date": pd.date_range(
            start=pd.to_datetime(daily.Time(), unit="s", utc=True),
            end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=daily.Interval()),
            inclusive="left"
        ).tz_convert("Asia/Bangkok").tz_localize(None)}

        for i, var in enumerate(daily_vars):
            daily_data[var] = daily.Variables(i).ValuesAsNumpy()

        df = pd.DataFrame(daily_data)
        
        # Thêm thông tin trạm
        df["station_id"] = st["station_id"]
        df["station_name"] = st["name"]
        df["ncei_latitude"] = st["lat"]
        df["ncei_longitude"] = st["lon"]
        df["latitude_om_actual"] = float(response.Latitude())
        df["longitude_om_actual"] = float(response.Longitude())

        all_df.append(df)
        print(f"  -> Thành công: {len(df)} ngày.")
        time.sleep(1) # Tránh bị chặn bởi server

    except Exception as e:
        print(f"  -> Lỗi tại {st['name']}: {e}")

if all_df:
    final_df = pd.concat(all_df, ignore_index=True)
    output_path = Path("/kaggle/working/openmeteo_daily_15stations.csv")
    final_df.to_csv(output_path, index=False)
    print(f"\nHOÀN THÀNH! Lưu tại: {output_path}")

## Crawl GSOD

In [ ]:
# Giữ nguyên dict trạm (GSOD vẫn dùng ID 11 ký tự này)
stations_dict = {
    "48820099999": "Hà Nội (Nội Bài)",
    "48823099999": "Hà Nội (Láng)",
    "48826099999": "Hải Phòng (Phù Liễn)",
    "48831099999": "Nam Định",
    "48840099999": "Thanh Hóa",
    "48845099999": "Nghệ An (Vinh)",
    "48848099999": "Thừa Thiên Huế (Phú Bài)",
    "48852099999": "Đà Nẵng",
    "48863099999": "Bình Định (Quy Nhơn)",
    "48870099999": "Khánh Hòa (Nha Trang)",
    "48866099999": "Lâm Đồng (Đà Lạt)",
    "48900099999": "TP. Hồ Chí Minh (Tân Sơn Nhất)",
    "48907099999": "Cần Thơ",
    "48894099999": "Đắk Lắk (Buôn Ma Thuột)",
    "48914099999": "Cà Mau",
}

all_df = []
start_year = 2015
end_year = 2024

print("--- BẮT ĐẦU DOWNLOAD GSOD DATASET TỪ NOAA NCEI ---")
start_time = time.time()

for ncei_id, st_name in stations_dict.items():
    print(f"\nĐang tải GSOD trạm: {st_name} (Mã: {ncei_id})")

    for year in range(start_year, end_year + 1):
        # Lưu ý: Thay đổi đường dẫn sang /gsod/
        url = f"https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/{year}/{ncei_id}.csv"

        try:
            df = pd.read_csv(url)
            
            # Thêm metadata
            df['ncei_station_id'] = ncei_id
            df['station_name']    = st_name
            
            all_df.append(df)
            print(f"  -> Năm {year}: Thành công - {len(df)} ngày.")

        except Exception as e:
            # GSOD đôi khi trống một số năm, đây là điều bình thường
            print(f"  -> Năm {year}: Không có dữ liệu hoặc lỗi kết nối.")

if all_df:
    print("\n--- ĐANG GỘP DỮ LIỆU GSOD ---")
    final_df = pd.concat(all_df, ignore_index=True)

    # Convert DATE sang datetime để dễ xử lý
    final_df['DATE'] = pd.to_datetime(final_df['DATE'])

    output_path = Path("/kaggle/working/ncei_gsod_daily.csv")
    final_df.to_csv(output_path, index=False)

    print(f"\nHOÀN THÀNH! Tổng số bản ghi ngày: {len(final_df):,}")
    print(f"File lưu tại: {output_path}")
    print(f"Cột quan trọng cần dùng: DATE, PRCP (lượng mưa), TEMP (nhiệt độ)")
else:
    print("\n[LỖI] Không tải được dữ liệu nào.")